In [6]:
import pandas as pd
from pathlib import Path
import csv
from collections import Counter

In [ ]:
# Comma diagnostic for transactions file
raw_path = Path("../data/raw/OmkarCapstone_Final_ArchivePassportData_20260123.csv")  
expected_commas = 13  # since there are 14 columns in the csv

with open(raw_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

# Count lines by comma count
from collections import Counter
comma_counts = Counter(line.count(',') for line in lines)

print("Comma count distribution:")
for count, freq in sorted(comma_counts.items()):
    flag = " expected" if count == expected_commas else " PROBLEM" if count > expected_commas else ""
    print(f"  {count} commas: {freq:,} lines{flag}")

# Show some sample problem lines
print("\nSample lines with extra commas")
problem_count = 0
for i, line in enumerate(lines[0:9000000]):  # to check for the entire file
    if line.count(',') > expected_commas:
        print(f"Line {i+1}: {line[:200]}...")
        problem_count += 1
        if problem_count >= 5:
            break

Comma count distribution:
  0 commas: 2 lines
  13 commas: 8,125,390 lines expected
  14 commas: 105,529 lines PROBLEM
  15 commas: 3,137 lines PROBLEM
  16 commas: 26 lines PROBLEM
  17 commas: 9 lines PROBLEM

Sample lines with extra commas
Line 106: WWW.ETSY.COM/,BROOKLYNNY,5699,ETSY, INC. ETSY.,20,POS Debit        Pri,P,Pinned Settlements  ,49,31.30,2025-11-22,143842, ,BSB189897
...
Line 729: 34 main st, bangor,,BangorME,5462,Mainely juice,0,POS Debit        Pri,P,Pinned Settlements  ,88,11.18,2025-12-19,123911, ,BSB217951
...
Line 731: 345 US Route 1, Sui,KitteryME,5655,NikePOS_US,0,POS Debit        Pri,P,Pinned Settlements  ,11,94.93,2025-12-21,183425, ,BSB105130
...
Line 745: WWW.ETSY.COM/,BROOKLYNNY,5699,ETSY, INC. ETSY.,20,POS Debit        Pri,P,Pinned Settlements  ,71,18.00,2025-11-22,40012, ,BSB398966
...
Line 1032: WWW.ETSY.COM/,BROOKLYNNY,5699,ETSY, INC. ETSY.,20,POS Debit        Pri,P,Pinned Settlements  ,38,33.17,2025-11-23,33305, ,BSB518066
...


In [8]:
# # Transaction data cleaning - to handle commas
# raw_path = Path("/home/omkar/ds5500/data/raw/OmkarCapstone_Final_ArchivePassportData_20260123.csv")
# clean_path = Path("/home/omkar/ds5500/data/processed/transactions_clean.csv")

# columns = [
#     "ATM_Address", "ATM_City_State", "Merchant_Category", "Merchant_Name",
#     "Transaction_Code", "Transaction_Code_Description", "Transaction_Type",
#     "Transaction_Type_Description", "Transaction_Number", "Amount_Completed",
#     "Transaction_Date", "Time_Local_hhmmss", "Recurring_Trxn", "CustomerID"
# ]

# def parse_line(line):
#     """Split from right (stable cols), then parse left 4 cols."""
#     line = line.strip()
#     parts = line.split(',')
    
#     if len(parts) == 14:
#         return parts  # clean line
    
#     # Rightmost 10 columns look stable - so split from right
#     right_10 = parts[-10:]
#     left_remainder = parts[:-10]  # should be 4+ parts for first 4 cols
    
#     # Reconstruct first 4: ATM_Address, ATM_City_State, Merchant_Category, Merchant_Name
#     # Merchant_Category is numeric (col 3), so find it and work around it
#     # Pattern: everything before the numeric MCC is address+city, after is merchant_name
    
#     for i, val in enumerate(left_remainder):
#         if val.strip().isdigit() and len(val.strip()) in [3, 4]:  # MCC codes are 3-4 digits
#             atm_address = ','.join(left_remainder[:i-1]) if i > 1 else left_remainder[0]
#             atm_city_state = left_remainder[i-1] if i > 0 else ''
#             merchant_category = val.strip()
#             merchant_name = ','.join(left_remainder[i+1:])
#             return [atm_address, atm_city_state, merchant_category, merchant_name] + right_10
    
#     # Fallback: couldn't parse, return None to skip
#     return None

# # Process file
# clean_path.parent.mkdir(parents=True, exist_ok=True)

# with open(raw_path, 'r', encoding='utf-8') as fin, \
#      open(clean_path, 'w', encoding='utf-8', newline='') as fout:
    
#     writer = csv.writer(fout)
#     writer.writerow(columns)  # header
    
#     success, failed = 0, 0
#     failed_samples = []
    
#     for i, line in enumerate(fin):
#         if not line.strip():
#             continue
        
#         parsed = parse_line(line)
#         if parsed:
#             writer.writerow(parsed)
#             success += 1
#         else:
#             failed += 1
#             if len(failed_samples) < 5:
#                 failed_samples.append((i+1, line[:150]))

# print(f"Cleaned: {success:,} rows")
# print(f"Failed:  {failed:,} rows")
# print(f"Output: {clean_path}")

# if failed_samples:
#     print("\nFailed samples")
#     for line_num, sample in failed_samples:
#         print(f"Line {line_num}: {sample}...")

In [ ]:
# Transaction data cleaning - to handle commas
raw_path = Path("../data/raw/OmkarCapstone_Final_ArchivePassportData_20260123.csv")
clean_path = Path("../data/processed/transactions_clean_v2.csv")

columns = [
    "ATM_Address", "ATM_City_State", "Merchant_Category", "Merchant_Name",
    "Transaction_Code", "Transaction_Code_Description", "Transaction_Type",
    "Transaction_Type_Description", "Transaction_Number", "Amount_Completed",
    "Transaction_Date", "Time_Local_hhmmss", "Recurring_Trxn", "CustomerID"
]

def parse_line(line):
    """
    Precision Parser: Uses MCC position to correctly identify City vs Address.
    Assumption: The field immediately BEFORE the MCC is always the City/State.
    """
    line = line.strip()
    parts = line.split(',')
    
    if len(parts) == 14:
        return parts 
    
    right_10 = parts[-10:]
    left_remainder = parts[:-10]
    
    # 1. FIND THE ANCHOR (Same robust logic as v6)
    mcc_index = None
    # We scan indices 2 to 6 to find the MCC
    for i in range(2, min(7, len(left_remainder))):
        val = left_remainder[i].strip()
        if val.isdigit() and len(val) in [3, 4]:
            mcc_index = i
            break
            
    if mcc_index is None:
        return None

    # 2. EXTRACT DATA RELATIVE TO THE ANCHOR
    
    # MCC is at the anchor index
    merchant_category = left_remainder[mcc_index].strip()
    
    # MERCHANT NAME is everything AFTER the anchor
    merchant_name = ','.join(left_remainder[mcc_index+1:])
    
    # CITY/STATE is the neighbor immediately LEFT of the anchor
    # (Since we search from index 2, index-1 is always safe)
    atm_city_state = left_remainder[mcc_index - 1].strip()
    
    # ADDRESS is everything LEFT of the City
    # We join it all back together (fixing the split street numbers)
    atm_address = ','.join(left_remainder[:mcc_index - 1])
    
    return [atm_address, atm_city_state, merchant_category, merchant_name] + right_10

# Process file
clean_path.parent.mkdir(parents=True, exist_ok=True)

with open(raw_path, 'r', encoding='utf-8') as fin, \
     open(clean_path, 'w', encoding='utf-8', newline='') as fout:
    
    writer = csv.writer(fout)
    writer.writerow(columns)  # header
    
    success, failed = 0, 0
    failed_samples = []
    
    for i, line in enumerate(fin):
        if not line.strip():
            continue
        
        parsed = parse_line(line)
        if parsed:
            writer.writerow(parsed)
            success += 1
        else:
            failed += 1
            if len(failed_samples) < 5:
                failed_samples.append((i+1, line[:150]))

print(f"Cleaned: {success:,} rows")
print(f"Failed:  {failed:,} rows")
print(f"Output: {clean_path}")

if failed_samples:
    print("\nFailed samples")
    for line_num, sample in failed_samples:
        print(f"Line {line_num}: {sample}...")

Cleaned: 8,234,091 rows
Failed:  1 rows
Output: /home/omkar/ds5500/data/processed/transactions_clean_v2.csv

Failed samples
Line 8234093: (8234091 rows affected)
...


In [ ]:
# Verify cleaned transactions file
clean_path = Path("../data/processed/transactions_clean_v2.csv")

with open(clean_path, 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    header = next(reader)
    
    print("Columns:", len(header))
    print(header)
    print("\nFirst 5 rows")
    for i, row in enumerate(reader):
        if i >= 5:
            break
        print(f"Row {i+1}: {len(row)} cols")
        for col, val in zip(header, row):
            print(f"  {col}: {val}")
        print()

Columns: 14
['ATM_Address', 'ATM_City_State', 'Merchant_Category', 'Merchant_Name', 'Transaction_Code', 'Transaction_Code_Description', 'Transaction_Type', 'Transaction_Type_Description', 'Transaction_Number', 'Amount_Completed', 'Transaction_Date', 'Time_Local_hhmmss', 'Recurring_Trxn', 'CustomerID']

First 5 rows
Row 1: 14 cols
  ATM_Address: 609653 BROADWAY
  ATM_City_State: BANGORME
  Merchant_Category: 5310
  Merchant_Name: TJMAXX #0433
  Transaction_Code: 20
  Transaction_Code_Description: POS Debit        Pri
  Transaction_Type: P
  Transaction_Type_Description: Pinned Settlements  
  Transaction_Number: 25
  Amount_Completed: 32.99
  Transaction_Date: 2025-11-23
  Time_Local_hhmmss: 141633
  Recurring_Trxn:  
  CustomerID: BSB149015

Row 2: 14 cols
  ATM_Address: HANNAFORD #8241 93
  ATM_City_State: BELFASTME
  Merchant_Category: 5411
  Merchant_Name: HANNAFORD #8241
  Transaction_Code: 20
  Transaction_Code_Description: POS Debit        Pri
  Transaction_Type: P
  Transaction_

In [ ]:
# Full validation of cleaned transactions file
clean_path = Path("../data/processed/transactions_clean_v2.csv")

col_counts = Counter()
mcc_issues = []
date_issues = []
customerid_issues = []

with open(clean_path, 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    header = next(reader)
    
    for i, row in enumerate(reader, start=2):  # line 2 onwards (after header)
        col_counts[len(row)] += 1
        
        # Spot check key columns (sample first 100 issues max)
        if len(row) == 14:
            mcc = row[2].strip()
            date = row[10].strip()
            cid = row[13].strip()
            
            if not mcc.isdigit() and len(mcc_issues) < 10:
                mcc_issues.append((i, mcc))
            if not date.startswith("202") and len(date_issues) < 10:
                date_issues.append((i, date))
            if not cid.startswith("BSB") and len(customerid_issues) < 10:
                customerid_issues.append((i, cid))

print("Column Count Distribution")
for count, freq in sorted(col_counts.items()):
    status = "Correct" if count == 14 else "Incorrect"
    print(f"  {status} {count} columns: {freq:,} rows")

total = sum(col_counts.values())
valid = col_counts.get(14, 0)
print(f"\nTotal: {total:,} | Valid: {valid:,} | Invalid: {total - valid:,}")

if mcc_issues:
    print(f"\nMCC issues (non-numeric): {mcc_issues[:5]}")
if date_issues:
    print(f"\nDate issues: {date_issues[:5]}")
if customerid_issues:
    print(f"\nCustomerID issues: {customerid_issues[:5]}")

if not mcc_issues and not date_issues and not customerid_issues and valid == total:
    print("\nFile is clean and ready for loading")

Column Count Distribution
  Correct 14 columns: 8,234,091 rows

Total: 8,234,091 | Valid: 8,234,091 | Invalid: 0

File is clean and ready for loading


Cleaning customers excel file

In [ ]:
raw_path = Path("../data/raw/OmkarCustomerDemographics.xlsx")
clean_path = Path("../data/processed/customers_clean.csv")

df = pd.read_excel(raw_path)
print(f"Rows: {len(df):,}")

# to remove rows which are completely empty
df = df.dropna(how="all")
print(f"After removing empty rows: {len(df):,}")

df.to_csv(clean_path, index = False)
print(f"Output: {clean_path}")

Rows: 276,840
After removing empty rows: 276,838
Output: /home/omkar/ds5500/data/processed/customers_clean.csv


In [13]:
# Compare Excel vs CSV output
df_excel = pd.read_excel(raw_path).dropna(how='all')
df_csv = pd.read_csv(clean_path)

print("Row counts -")
print(f"Excel: {len(df_excel):,}")
print(f"CSV:   {len(df_csv):,}")

print("\nColumn types -")
print(f"{'Column':<30} {'Excel':<15} {'CSV':<15}")
for col in df_excel.columns:
    print(f"{col:<30} {str(df_excel[col].dtype):<15} {str(df_csv[col].dtype):<15}")

print("\nFirst 3 rows (Excel)")
print(df_excel.head(3).to_string())

print("\nFirst 3 rows (CSV)")
print(df_csv.head(3).to_string())

Row counts -
Excel: 276,838
CSV:   276,838

Column types -
Column                         Excel           CSV            
CustomerID                     str             str            
Individual                     str             str            
Age                            float64         float64        
Gender                         str             str            
NAICSCode                      float64         float64        
OriginalCustomerDate           datetime64[us]  str            
RelationshipYears              float64         float64        
RelationshipMonths             float64         float64        
BangorWealth                   str             str            
Payroll                        str             str            
Merchant                       str             str            
TPS                            str             str            
PlingCustomer                  str             str            
ABLECustomer                   str             str         